# Importing libraries, loading and transforming data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# !pip install -q evaluate transformers==4.28.1
# !pip install -U -q datasets
# !pip install -q torchaudio==0.12
# !add-apt-repository -y ppa:savoury1/ffmpeg4
# !apt-get -qq install -y ffmpeg
# !pip install -q mlflow

In [ ]:
#imports
import pandas as pd
import gc
import re
import numpy as np
import os

import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm
tqdm.pandas()
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import torch
import datasets
import transformers
print(transformers.__version__)

In [4]:
# df_list = []
# for loc in ['Train', 'Test']:
#     df_tmp = pd.read_csv(f"/kaggle/input/musical-instruments-sound-dataset/Metadata_{loc}.csv")
#     df_list.append(df_tmp)
# df = pd.concat(df_list, axis=0)
# df['Class'] = df['Class'].replace({"Sound_Guiatr": "Sound_Guitar"})
# # remove violin label as it is contaminated by Drums
# df = df[df['Class']!="Sound_Violin"]
# print(df.shape)
# df.sample(5).T

In [5]:
import os
import pandas as pd

directory = r'/content/drive/MyDrive/music_dataset'

train_df = pd.DataFrame(columns = ['Class', 'FileName','file'])
test_df = pd.DataFrame(columns = ['Class', 'FileName','file'])

for folder in os.listdir(directory):
    folder_path = os.path.join(directory, folder)
    for file in os.listdir(folder_path):
        if int(file[:-4]) < 160:
            file_path = os.path.join(folder_path, file)
            train_df = pd.concat([train_df, pd.DataFrame([{'Class': folder, 'FileName': file, 'file':file_path}])], ignore_index=True)
        else:
            file_path = os.path.join(folder_path, file)
            test_df = pd.concat([test_df, pd.DataFrame([{'Class': folder, 'FileName': file, 'file':file_path}])], ignore_index=True)


In [6]:
df_list = []
df_list.append(train_df)
df_list.append(test_df)
df = pd.concat(df_list, axis=0)

In [ ]:
print(df.shape)
df.sample(5).T

In [ ]:
df['Class'].value_counts()

In [ ]:
RATE_HZ = 16000 # resampling rate in Hz
MAX_LENGTH = 48000 # maximum audio interval length to consider (= RATE_HZ * SECONDS)
labels = ['Acoustic_Guitar',
'Bass_Guitar',
'Drum_set',
'Electro_Guitar',
'flute',
'Hi_Hats',
'Keyboard',
'Trumpet',
'Violin']

label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

print(id2label, '\n\n', label2id)

In [10]:
from pathlib import Path
import torchaudio

# def load_data():
#     file_list = []
#     sample_list = []
#     for file in Path('/kaggle/input/musical-instruments-sound-dataset/').glob('*_submission/*_submission/*.wav'):
#         sample_name = file.stem + '.wav' #re.match(r"(\d+)", file.stem).group(0)
#         file_list.append(file)
#         sample_list.append(str(sample_name))
#     dd = pd.DataFrame()
#     dd['file'] = file_list
#     dd['FileName'] = sample_list
#     return dd

In [ ]:
dd = df.copy()
# dd = dd.set_index('FileName').join(df.set_index('FileName'), how='inner')
dd = dd[dd['Class'].isin(labels)]
dd['label'] = dd['Class'].apply(lambda x: label2id[x])
selected_cols = ['file', 'label', 'Class']
dd = dd[selected_cols]
dd.sample(5).T

In [ ]:
dd = dd.reset_index(drop=True)
dd.head()

In [ ]:
def get_transform_audio(file):
    audio,rate = torchaudio.load(str(file))
    transform = torchaudio.transforms.Resample(rate,RATE_HZ)
    audio = transform(audio).squeeze(0).numpy()
    audio = audio[:MAX_LENGTH] # truncate to first part of audio to save RAM
    # return audio only if it is full length audio
    if audio.shape[0]==MAX_LENGTH:
        return audio
dd['audio'] = dd['file'].progress_apply(get_transform_audio)

In [ ]:
# dd = dd.dropna(subset=['audio'])
dd.shape

In [ ]:
dd.isnull().sum()


In [ ]:
# %%time
# random oversampling of all minority classes
y = dd[['label']]
dd = dd.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
dd, y_resampled = ros.fit_resample(dd, y)
del y
dd['label'] = y_resampled
del y_resampled
gc.collect()

In [ ]:
dd.shape, dd['label'].value_counts()

In [18]:
# %%time
dd = dd.drop(['file'], axis=1)

In [ ]:
dd.sample(5).T

In [20]:
from datasets import Dataset
dd = Dataset.from_pandas(dd)

In [ ]:
from collections import Counter
Counter(dd['label']).items()

In [ ]:
dd = dd.train_test_split(test_size=0.2)
dd

# Load facebook/wav2vec2-base-960h model

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

model_str = "facebook/wav2vec2-base-960h"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_str)
model = AutoModelForAudioClassification.from_pretrained(model_str,num_labels=len(labels))
model.config.id2label = id2label
# number of trainable parameters
print(model.num_parameters(only_trainable=True)/1e6)

In [ ]:
def preprocess_function(batch):
    inputs = feature_extractor(batch['audio'], sampling_rate=RATE_HZ, max_length=MAX_LENGTH, truncation=True)
    inputs['input_values'] = inputs['input_values'][0]
    return inputs

dd['test'] = dd['test'].map(preprocess_function, remove_columns="audio", batched=False)
dd['train'] = dd['train'].map(preprocess_function, remove_columns="audio", batched=False)

In [ ]:
gc.collect()

# Train and evaluate model

In [26]:
import evaluate

accuracy = evaluate.load("accuracy")

from sklearn.metrics import roc_auc_score
def compute_metrics(eval_pred):
    # Compute the ROC AUC score
    predictions = eval_pred.predictions
    predictions = np.exp(predictions)/np.exp(predictions).sum(axis=1, keepdims=True)
    label_ids = eval_pred.label_ids
    roc_auc = roc_auc_score(label_ids, predictions, average='macro', multi_class='ovr') # one-vs-rest ROC AUC score

    # Calculate accuracy using the loaded accuracy metric
    acc_score = accuracy.compute(predictions=predictions.argmax(axis=1), references=label_ids)['accuracy']

    return {
        "roc_auc": roc_auc,
        "accuracy": acc_score
    }

In [27]:
from transformers import TrainingArguments, Trainer
batch_size=1
warmup_steps=50
weight_decay=0.02
num_train_epochs=5
model_name = "epoch_musical_instruments_identification_2"
training_args = TrainingArguments(
    output_dir=model_name,
    logging_dir='./logs',
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=5e-6, # 1e-6
    logging_strategy='steps',
    logging_first_step=True,
    load_best_model_at_end=True,
    logging_steps=1,
    evaluation_strategy='epoch',
    warmup_steps=warmup_steps,
    weight_decay=weight_decay,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    eval_steps=1,
    save_strategy='epoch',
    save_total_limit=1, # save fewer checkpoints to limit used space
    report_to="mlflow",  # log to mlflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.evaluate()

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [31]:
trainer.save_model()

# Send model to Huggingface

In [ ]:
# finally, save the model to Huggingface
from huggingface_hub import notebook_login
notebook_login()

In [40]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = f"Bhaveen/{model_name}"
try:
    api.create_repo(repo_id)
except:
    print(f"Repo {repo_id} already exists")

In [ ]:
api.upload_folder(
    folder_path=model_name,
    path_in_repo = ".",
    repo_id=repo_id,
    repo_type="model"
)